In [1]:
import os

In [2]:
%pwd

'd:\\PredictBot-Score-MLOps\\research'

In [3]:
os.chdir('D:\PredictBot-Score-MLOps')

In [4]:
%pwd

'D:\\PredictBot-Score-MLOps'

In [10]:
from dataclasses import dataclass
from pathlib import Path
from src.predictor_bot_score.config.configuration import yaml_load , create_directories
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.constants import CONFIG_PATH
from src.predictor_bot_score.utils.model_factory import get_model , get_fit_kwargs , get_mlflow_logger , MODEL_REGISTRY
from sklearn.metrics import mean_absolute_error
import os
import json
import numpy as np
import pandas as pd
import glob
import mlflow
import lightgbm
import pickle
from datetime import datetime
import sqlite3
import gc


In [6]:
@dataclass(frozen=True)
class ModelTrainingConfig:
    train_data_path       : Path
    val_data_path         : Path
    model_dir             : Path
    active_model_strategy : str
    models                : dict
    features              : list[str]
    target_column         : str
    baseline_mae          : float
    promotion_criteria    : dict
    mlflow_experiment     : str
    mlflow_tracking_uri   : str

In [7]:
class config_manager:

    def __init__(self, config = CONFIG_PATH):

        self.config = yaml_load(config)
        
        create_directories([self.config.artifacts_root])

    def get_model_training_config(self) -> ModelTrainingConfig:

        config = self.config.model_training

        create_directories([config.model_dir])

        return ModelTrainingConfig(
        train_data_path       = Path(config.train_data_path),
        val_data_path         = Path(config.val_data_path),
        model_dir             = Path(config.model_dir),
        active_model_strategy = config.active_model_strategy,
        models                = dict(config.models),
        features              = list(config.features),
        target_column         = config.target_column,
        baseline_mae          = float(config.baseline_mae),
        promotion_criteria    = dict(config.promotion_criteria),
        mlflow_experiment     = config.mlflow.experiment_name,
        mlflow_tracking_uri   = config.mlflow.tracking_uri,
    )
        

In [ ]:


class Model_Building :

    def __init__(self, config : ModelTrainingConfig):
        self.config = config
        self.train , self.val = self._read_data()
        self.batch_id = datetime.now().strftime("%Y_%m_%d_%H") 

    def _get_files(self , folder:Path ,prefix :str):
        files = glob.glob(os.path.join(folder ,f"{prefix}_*.csv"))
        if not files:
            raise FileNotFoundError(f"No {prefix} file found in {folder}")
        return max(files , key=os.path.getmtime)
    

    def _read_data(self):
        try:
            logger.info("=" * 50)
            logger.info("Reading train /  test splits")
            logger.info("=" * 50)

            train_file = self._get_files(self.config.train_data_path, "train")
            val_file   = self._get_files(self.config.val_data_path,   "val")

            train = pd.read_csv(train_file)
            val   = pd.read_csv(val_file)
            
            logger.info(f"Train : {len(train)} rows")
            logger.info(f"Val   : {len(val)} rows")

            return train, val

        except FileNotFoundError as e:
            logger.error(f"Split file not found: {e}")
            raise

        except Exception as e:
            logger.error(f"Failed to read splits: {str(e)}")
            raise

    def prepare_data(self ,df :pd.DataFrame):
        try:
            X = df[self.config.features]
            y = df[self.config.target_column]

            return X ,y
        except Exception as e:
            raise e 
        
    def _smape(self, actual , predicted ):
        try:
            sampe_result = float(
                100 * np.mean(
                    2 * np.abs(predicted - actual) /
                    (np.abs(actual) + np.abs(predicted) + 1e-8)
                )
            )

            return sampe_result
        except Exception as e:
            logger.error(f"SMAPE calculation failed: {str(e)}")
            raise 
    
    def model_training(self):
        try:
            X_train, y_train = self.prepare_data(self.train)
            X_val,   y_val   = self.prepare_data(self.val)

            trained_models = {}

            for model_name, cfg in self.config.models.items():
                model_type   = cfg["type"]
                model_params = cfg["params"]

                if model_type not in MODEL_REGISTRY:
                    logger.warning(
                        f"Skipping {model_name}: '{model_type}' not in registry"
                    )
                    continue

                logger.info("")
                logger.info(f"--- Training {model_name} ({model_type}) ---")
                logger.info("-" * 50)

                model      = get_model(model_type, model_params)
                eval_set   = [(X_val, y_val)]
                fit_kwargs = get_fit_kwargs(model_type, eval_set)

                model.fit(X_train, y_train, **fit_kwargs)


                trained_models[model_name] = {
                    "model"      : model,
                    "model_type" : model_type,
                    "params"     : model_params
                }

                logger.info(f"PASSED - {model_name} trained")

            return trained_models

        except Exception as e:
            logger.error(f"Training failed: {str(e)}")
            raise
    
    def save_model(self,model , model_name):

        try:
            path = self.config.model_dir
            time_stamp = datetime.now().strftime("%Y_%m_%d_%H")
            

            run_dir = os.path.join(path ,f'model_trained__{time_stamp}')
            os.makedirs(run_dir , exist_ok=True)

            model_path = os.path.join(run_dir,f"{model_name}.pkl")
            
            with open(model_path ,"wb") as f:
                pickle.dump(model , f)
                f.close()
        
            logger.info(f"Model saved  {model_path}")
            logger.info("PASSED - Model saved")
            logger.info("-" * 50)

            return model_path

        except Exception as e:
            logger.error(f"Failed to save model: {str(e)}")
            raise

    # ── log to mlflow ─────────────────────────────────────

    def save_report(self, report):

        try:
            logger.info("")
            logger.info("STEP - SAVING TRAINING REPORT")
            logger.info("-" * 50)

            if not report:
                raise ValueError("Report is empty — nothing to save")
            
            time_stamp = datetime.now().strftime("%Y_%m_%d_%H")
            report_path = os.path.join(self.config.model_dir , f"Model_training_report__{time_stamp}.json")

            with open(report_path, 'w') as f:
                json.dump(report, f, indent=4)

            logger.info(f"Report saved -> {report_path}")
            logger.info("PASSED - Training report saved successfully")
            return report_path

        except Exception as e:
            logger.error(f"Failed to save training report: {str(e)}")
            raise


    def run(self):
        trained_models = None

        try:
            logger.info("=" * 50)
            logger.info(f'{"=" * 50},MODEL TRAINING PIPELINE STARTED')
            logger.info("=" * 50)

            trained_models = self.model_training()

            logger.info("")
            logger.info("=" * 50)
            logger.info("TRAINING SUMMARY")
            logger.info("=" * 50)
            

            logger.info("")
            logger.info("=" * 50)
            logger.info("SAVING AND LOGGING ALL MODELS")
            logger.info("=" * 50)

            report = {}
            for model_name, result in trained_models.items():
                
                model_path = self.save_model(result["model"], model_name)
                report[model_name] = {
                "model_type"  : result["model_type"],
                "params"      : result["params"],
                "model_path"  : model_path,
                "timestamp"   : datetime.now().isoformat()
            }

            self.save_report(report) ## this function creates the model which acts as the input for mext self.log_mlflow
                
            logger.info("")
            logger.info("=" * 50)
            logger.info("MODEL TRAINING PIPELINE COMPLETE")
            logger.info("=" * 50)

            return None

        except Exception as e:
            logger.error(f"Model training pipeline failed: {str(e)}")
            raise

        finally:
            if trained_models is not None:
                for res in trained_models.values():
                    res["model"] = None
                trained_models = None

            self.train = None
            self.val   = None
            gc.collect()
            logger.info("Memory cleared")
            logger.info(f'{"=" * 50},MODEL TRAINING PIPELINE COMPLETED')



             

   



            

   

In [9]:
xc = config_manager()
xc = xc.get_model_training_config()
xc = Model_Building(xc)
xc.model_training()


[2026-06-27 17:05:38,743: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-27 17:05:38,751: INFO: common: Directory created (or already exists) at: artifacts]
[2026-06-27 17:05:38,754: INFO: common: Directory created (or already exists) at: artifacts/Model_Building]
[2026-06-27 17:05:38,755: INFO: 514148386: ==================================================]
[2026-06-27 17:05:38,757: INFO: 514148386: Reading train /  test splits]
[2026-06-27 17:05:38,758: INFO: 514148386: ==================================================]
[2026-06-27 17:05:38,947: INFO: 514148386: Train : 28426 rows]
[2026-06-27 17:05:38,949: INFO: 514148386: Val   : 4061 rows]
[2026-06-27 17:05:38,968: INFO: 514148386: ]
[2026-06-27 17:05:38,968: INFO: 514148386: --- Training lightgbm (lightgbm) ---]
[2026-06-27 17:05:38,968: INFO: 514148386: --------------------------------------------------]
[200]	valid_0's l1: 0.019045
[400]	valid_0's l1: 0.0172147
[600]	valid_0's l1: 0.0169749
[800]	vali

{'lightgbm': {'model': LGBMRegressor(colsample_bytree=0.9, learning_rate=0.01, max_depth=8,
                metric='mae', min_child_samples=30, n_estimators=2000,
                num_leaves=63, objective='regression', random_state=42,
                reg_alpha=0.5, reg_lambda=0.5, subsample=0.9, verbose=-1),
  'model_type': 'lightgbm',
  'params': ConfigBox({'objective': 'regression', 'metric': 'mae', 'n_estimators': 2000, 'learning_rate': 0.01, 'num_leaves': 63, 'max_depth': 8, 'min_child_samples': 30, 'subsample': 0.9, 'colsample_bytree': 0.9, 'reg_alpha': 0.5, 'reg_lambda': 0.5, 'random_state': 42, 'verbose': -1})},
 'xgboost': {'model': XGBRegressor(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=0.9, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
     

In [39]:
yaml = yaml_load(Path('config\config.yaml'))

[2026-06-25 11:59:11,661: INFO: common: yaml file: config\config.yaml loaded successfully]


In [31]:
yaml.model_training.mlflow.experiment_name

'predict-bot-training'